# ЕГРОКН → вырезка по Республике Алтай

Источник: выгрузка ЕГРОКН (data-70-structure-16.csv, 516 МБ, вся Россия, получена 28.10.2026).
Задача: отфильтровать Республику Алтай, проверить координаты и категории, сохранить в `data/okn/`.

Файл большой — читаем по кускам (chunks), оставляя только нужные колонки.

In [1]:
import pandas as pd
from pathlib import Path

SRC = Path('/home/mfthesun/Workspace/Пространственное планирование/data-70-structure-16.csv')
# быстрый взгляд на заголовок
cols = pd.read_csv(SRC, nrows=0, sep=',', encoding='utf-8').columns.tolist()
print(len(cols), 'колонок')
print(cols)

56 колонок
['Объект', 'Номер в реестре', 'название улицы', 'Комментарий', 'Идентификатор дома в ФИАС', 'FiasAuxSubobjId', 'FiasAuxObjId', 'Идентификатор улицы в ФИАС', 'Идентификатор населенного пункта в ФИАС', 'Идентификатор внутрегородской территории в ФИАС', 'Идентификатор города в ФИАС', 'Идентификатор городских и сельских поселений в ФИАС', 'Идентификатор района в ФИАС', 'Идентификатор автономного округа в ФИАС', 'Идентификатор региона в ФИАС', 'Идентификатор страны в ФИАС', 'Полный адрес', 'Адрес постановки', 'На карте', 'Id-Регион', 'Регион', 'учетный номер', 'Id-Категория историко-культурного значения', 'Категория историко-культурного значения', 'Id-Тип объекта культурного наследия', 'Тип объекта культурного наследия', 'Id-Вид объекта', 'Вид объекта', 'описание предмета охраны', 'текстовое описание границ', 'Id-общая видовая принадлежность', 'Id-Принадлежность к Юнеско', 'Принадлежность к Юнеско', 'Id-Особо ценный объект', 'Особо ценный объект', 'ВОВ-id', 'Принадлежность к ВОВ'

In [2]:
KEEP = ['Объект', 'Номер в реестре', 'Полный адрес', 'Id-Регион', 'Регион',
        'Категория историко-культурного значения', 'Тип объекта культурного наследия',
        'Вид объекта', 'Принадлежность к Юнеско', 'дата создания', 'GeoId',
        'X', 'Y', 'наименование документа', 'признак актуальности', 'признак публичности',
        'год постановки ОКН на Госохрану', 'Id - идентификатор']

parts, n_total = [], 0
for ch in pd.read_csv(SRC, usecols=lambda c: c in KEEP, chunksize=200_000,
                      dtype=str, encoding='utf-8'):
    n_total += len(ch)
    parts.append(ch[ch['Id-Регион'].astype(str).str.zfill(2) == '04'])
ra = pd.concat(parts, ignore_index=True)
print(f'всего строк в выгрузке: {n_total:,}; строк по Республике Алтай: {len(ra):,}')

всего строк в выгрузке: 163,701; строк по Республике Алтай: 288


In [3]:
print('Категория:'); print(ra['Категория историко-культурного значения'].value_counts(dropna=False))
print('\nТип ОКН:'); print(ra['Тип объекта культурного наследия'].value_counts(dropna=False))
print('\nАктуальность/публичность:'); print(ra['признак актуальности'].value_counts(dropna=False), ra['признак публичности'].value_counts(dropna=False))
print('\nЮНЕСКО:'); print(ra['Принадлежность к Юнеско'].value_counts(dropna=False))

Категория:
Категория историко-культурного значения
Регионального значения                170
Федерального значения                 117
Местного (муниципального) значения      1
Name: count, dtype: int64

Тип ОКН:
Тип объекта культурного наследия
NaN    288
Name: count, dtype: int64

Актуальность/публичность:
признак актуальности
true    288
Name: count, dtype: int64 признак публичности
true    171
NaN     117
Name: count, dtype: int64

ЮНЕСКО:
Принадлежность к Юнеско
нет    288
Name: count, dtype: int64


In [4]:
xy = ra[['X', 'Y']].apply(pd.to_numeric, errors='coerce')
print('X заполнено:', xy['X'].notna().sum(), 'из', len(ra), '; Y заполнено:', xy['Y'].notna().sum())
ok = xy['X'].between(84, 91) & xy['Y'].between(49, 53)   # границы РА с запасом
print('координаты внутри РА (84-91 E, 49-53 N):', int(ok.sum()))
print('X/Y диапазоны:', xy['X'].min(), xy['X'].max(), '|', xy['Y'].min(), xy['Y'].max())
print('дубликаты по «Номер в реестре»:', int(ra['Номер в реестре'].duplicated().sum()))

X заполнено: 0 из 288 ; Y заполнено: 0
координаты внутри РА (84-91 E, 49-53 N): 0
X/Y диапазоны: nan nan | nan nan
дубликаты по «Номер в реестре»: 0


In [5]:
# «На карте» может содержать ссылку с координатами — проверяем (перечитываем с доп. колонками)
KEEP2 = KEEP + ['На карте', 'GeoId']
parts = []
for ch in pd.read_csv(SRC, usecols=lambda c: c in KEEP2, chunksize=200_000, dtype=str, encoding='utf-8'):
    parts.append(ch[ch['Id-Регион'].astype(str).str.zfill(2) == '04'])
ra2 = pd.concat(parts, ignore_index=True)
print(ra2['На карте'].dropna().head(5).tolist())
print('непустых «На карте»:', ra2['На карте'].notna().sum(), 'из', len(ra2))

['{"coordinates":[85.620388,51.416586],"type":"Point"}', '{"coordinates":[85.680647,51.288083],"type":"Point"}', '{"coordinates":[85.676111,51.294155],"type":"Point"}', '{"coordinates":[85.751048,51.725318],"type":"Point"}', '{"coordinates":[85.780971,51.825035],"type":"Point"}']
непустых «На карте»: 76 из 288


In [6]:
import json
ra = ra2.copy()
def parse_pt(s):
    if isinstance(s, str) and s.startswith('{'):
        try:
            d = json.loads(s)
            lon, lat = d['coordinates']
            return lon, lat
        except Exception:
            return None, None
    return None, None
pts = ra['На карте'].apply(parse_pt)
ra['lon'] = [p[0] for p in pts]
ra['lat'] = [p[1] for p in pts]
print('с координатами:', ra['lon'].notna().sum())

# район из «Полный адрес»: «Республика Алтай, <район>, ...»
import re
def parse_mo(addr):
    if not isinstance(addr, str):
        return None
    m = re.match(r'Республика Алтай,\s*([^,]+)', addr)
    return m.group(1).strip() if m else None
ra['район (по адресу)'] = ra['Полный адрес'].apply(parse_mo)
print(ra['район (по адресу)'].value_counts(dropna=False))

с координатами: 76
район (по адресу)
NaN                       119
Усть-Коксинский район      26
Усть-Канский район         20
Онгудайский район          20
Кош-Агачский район         19
Шебалинский район          16
Чемальский район           15
Майминский район           13
Турочакский район          12
Улаганский район           11
Чойский район               8
г. Горно-Алтайск            5
город Горно-Алтайск         2
МО "Шебалинский район"      1
Улагансикй район            1
Name: count, dtype: int64


In [7]:
kat = ra['Категория историко-культурного значения'].str.replace(' (муниципального)', '', regex=False).str.replace('значения', 'зн.', regex=False)
agg = (ra.assign(кат=kat)
         .pivot_table(index='район (по адресу)', columns='кат', values='Объект',
                      aggfunc='count', fill_value=0))
print(agg)
print('\nИтого по категориям:', kat.value_counts().to_dict())

кат                     Местного зн.  Регионального зн.
район (по адресу)                                      
Кош-Агачский район                 0                 19
МО "Шебалинский район"             0                  1
Майминский район                   0                 13
Онгудайский район                  0                 20
Турочакский район                  0                 12
Улагансикй район                   0                  1
Улаганский район                   0                 11
Усть-Канский район                 0                 20
Усть-Коксинский район              0                 26
Чемальский район                   0                 15
Чойский район                      0                  8
Шебалинский район                  1                 15
г. Горно-Алтайск                   0                  5
город Горно-Алтайск                0                  2

Итого по категориям: {'Регионального зн.': 170, 'Федерального зн.': 117, 'Местного зн.': 1}


In [8]:
nan_addr = ra.loc[ra['район (по адресу)'].isna(), ['Полный адрес', 'Категория историко-культурного значения']]
print('пустых адресов:', nan_addr['Полный адрес'].isna().sum())
print(nan_addr['Полный адрес'].dropna().head(10).tolist())
print('\nфедеральных без района:', (nan_addr['Категория историко-культурного значения'] == 'Федерального значения').sum())

пустых адресов: 118
['Вблизи жилого дома, находящегося по адресу: Республика Алтай, Кош-Агачский район, село Тобелер, улица Советская, 19']

федеральных без района: 117


In [9]:
# нормализация района: опечатки и варианты написания
import re
fix = {'Улагансикй район': 'Улаганский район', 'МО "Шебалинский район"': 'Шебалинский район',
       'город Горно-Алтайск': 'г. Горно-Алтайск'}
ra['район'] = ra['район (по адресу)'].replace(fix)

# у кого район не распознался — ищем название района по всему адресу
mo_names = ['Кош-Агачский', 'Майминский', 'Онгудайский', 'Турочакский', 'Улаганский',
            'Усть-Канский', 'Усть-Коксинский', 'Чемальский', 'Чойский', 'Шебалинский']
def find_mo(row):
    if isinstance(row['район'], str):
        return row['район']
    addr = row['Полный адрес']
    if isinstance(addr, str):
        for m in mo_names:
            if m in addr:
                return m + ' район'
        if 'Горно-Алтайск' in addr:
            return 'г. Горно-Алтайск'
    return 'район не определён'
ra['район'] = ra.apply(find_mo, axis=1)
print(ra['район'].value_counts())

район
район не определён       118
Усть-Коксинский район     26
Кош-Агачский район        20
Усть-Канский район        20
Онгудайский район         20
Шебалинский район         17
Чемальский район          15
Майминский район          13
Турочакский район         12
Улаганский район          12
Чойский район              8
г. Горно-Алтайск           7
Name: count, dtype: int64


In [11]:
from pathlib import Path
REPO = Path('/home/mfthesun/Projects/altai-spatial-planning')
fed_stp = pd.read_csv(REPO / 'data/stp_extracted/okn/okn_federal.csv')
print(fed_stp.columns.tolist()); print(len(fed_stp)); fed_stp.head(3)

['№ п/п', '№ п/п_2', 'Учетный номер', 'Наименование', 'Вид ОКН', 'Входит в ансамбль', 'Общая видовая принадлежность', 'Наименование нормативно-правового акта о постановке на государственную охрану', 'Тип процесса/ статус', 'Регистрационный номер', 'section']
118


,№ п/п,№ п/п_2,Учетный номер,Наименование,Вид ОКН,Входит в ансамбль,Общая видовая принадлежность,Наименование нормативно-правового акта о постановке на государственную охрану,Тип процесса/ статус,Регистрационный номер,section
0,1,NaN,04-113035,Каменное изваяние,Памятник,NaN,Памятник археологии,Постановление Совета Министров РСФСР от 30.08....,Регистрация**,041740777030006,Город Горно-Алтайск
1,2,NaN,04-76325,"Каменное изваяние ""Кезер""",Памятник,NaN,Памятник археологии,Постановление Совета Министров РСФСР от 30.08....,Регистрация,041640441370006,Город Горно-Алтайск
2,3,NaN,04-59982,Улалинская стоянка,Памятник,NaN,Памятник археологии,Указ Президента Российской Федерации от 20.02....,Регистрация,041540298210006,Город Горно-Алтайск


In [13]:
# в СТП есть и «Регистрационный номер» — связываем по нему
m = ra.merge(fed_stp[['Регистрационный номер', 'section']].drop_duplicates('Регистрационный номер'),
             left_on='Номер в реестре', right_on='Регистрационный номер', how='left')
print('совпало по рег. номеру:', m['section'].notna().sum(), 'из', len(ra))
print(m.loc[m['section'].notna(), 'section'].value_counts())
print('\nне совпали (федеральные):', ((m['section'].isna()) & (m['Категория историко-культурного значения'] == 'Федерального значения')).sum())

совпало по рег. номеру: 115 из 288
section
Онгудайский муниципальный район        41
Кош-Агачский муниципальный район       30
Майминский муниципальный район          9
Чемальский муниципальный район          9
Улаганский муниципальный район          9
Усть-Канский муниципальный район        6
Шебалинский муниципальный район         4
Город Горно-Алтайск                     3
Турочакский муниципальный район         3
Усть-Коксинский муниципальный район     1
Name: count, dtype: int64

не совпали (федеральные): 2


In [14]:
print('муниципальный объект:')
print(m.loc[m['Категория историко-культурного значения'].str.contains('Местного'), ['Объект', 'Полный адрес', 'Номер в реестре']].to_string())
print('\nнесовпавшие федеральные:')
print(m.loc[(m['section'].isna()) & (m['Категория историко-культурного значения'] == 'Федерального значения'), ['Объект', 'Номер в реестре', 'наименование документа']].to_string())

муниципальный объект:
                                       Объект                                                                                                                 Полный адрес  Номер в реестре
80  Территория бывшего села Кумжулу (Камышла)  Республика Алтай, Шебалинский район, Камлакское сельское поселение, в границах ликвидированного населенного пункта Камышла   041931331800004

несовпавшие федеральные:
                                                              Объект  Номер в реестре наименование документа
143  Курганная группа "Кызыл-Телань-1", I тыс. до н.э. - I тыс. н.э.  041640508970046                     []
160                                                  Группа курганов  041740876180026                     []


In [15]:
# финальный район: адрес -> СТП-присоединение для федеральных
mo_map = {'Онгудайский муниципальный район': 'Онгудайский район', 'Кош-Агачский муниципальный район': 'Кош-Агачский район',
          'Майминский муниципальный район': 'Майминский район', 'Чемальский муниципальный район': 'Чемальский район',
          'Улаганский муниципальный район': 'Улаганский район', 'Усть-Канский муниципальный район': 'Усть-Канский район',
          'Шебалинский муниципальный район': 'Шебалинский район', 'Город Горно-Алтайск': 'г. Горно-Алтайск',
          'Турочакский муниципальный район': 'Турочакский район', 'Усть-Коксинский муниципальный район': 'Усть-Коксинский район'}
stp_mo = m['section'].map(mo_map)
m['район'] = m['район'].where(m['район'] != 'район не определён', stp_mo)
print(m['район'].value_counts(dropna=False))
print('\nбез района осталось:', m['район'].isna().sum())

район
Онгудайский район        61
Кош-Агачский район       50
Усть-Коксинский район    27
Усть-Канский район       26
Чемальский район         24
Майминский район         22
Шебалинский район        21
Улаганский район         21
Турочакский район        15
г. Горно-Алтайск         10
Чойский район             8
NaN                       3
Name: count, dtype: int64

без района осталось: 3


In [16]:
OUT = REPO / 'data/okn'
OUT.mkdir(exist_ok=True)

out = m.rename(columns={'Объект': 'название', 'Номер в реестре': 'рег_номер',
                        'Категория историко-культурного значения': 'категория',
                        'Вид объекта': 'вид', 'Полный адрес': 'адрес',
                        'наименование документа': 'документ_охраны',
                        'Принадлежность к Юнеско': 'юнеско', 'дата создания': 'дата_создания',
                        'признак актуальности': 'актуальность'})[
    ['название', 'категория', 'район', 'вид', 'адрес', 'рег_номер', 'документ_охраны',
     'юнеско', 'дата_создания', 'актуальность', 'lon', 'lat']]
out['категория'] = out['категория'].str.replace('историко-культурного значения', 'значения', regex=False)
out = out.sort_values(['категория', 'район', 'название']).reset_index(drop=True)
out.to_csv(OUT / 'egrokn_altai.csv', index=False, encoding='utf-8-sig')
print('CSV:', len(out), 'строк →', OUT / 'egrokn_altai.csv')

# точки с координатами → geojson
pts = out.dropna(subset=['lon', 'lat'])
import geopandas as gpd
from shapely.geometry import Point
gdf = gpd.GeoDataFrame(pts, geometry=[Point(x, y) for x, y in zip(pts['lon'], pts['lat'])], crs=4326)
gdf = gdf.drop(columns=['lon', 'lat'])
gdf.to_file(OUT / 'egrokn_altai_points.geojson', driver='GeoJSON', encoding='utf-8')
print('GeoJSON:', len(gdf), 'точек →', OUT / 'egrokn_altai_points.geojson')

CSV: 288 строк → /home/mfthesun/Projects/altai-spatial-planning/data/okn/egrokn_altai.csv
GeoJSON: 76 точек → /home/mfthesun/Projects/altai-spatial-planning/data/okn/egrokn_altai_points.geojson


In [18]:
kat_map = {'Федерального значения': 'федеральные', 'Регионального значения': 'региональные',
           'Местного (муниципального) значения': 'местные'}
tab = out.assign(**{'значение': out['категория'].map(kat_map),
                    'район2': out['район'].fillna('район не определён')})
svod = tab.pivot_table(index='район2', columns='значение', values='название', aggfunc='count', fill_value=0)
svod = svod[['федеральные', 'региональные', 'местные']]
svod['итого'] = svod.sum(axis=1)
svod.loc['Итого'] = svod.sum()
svod.to_csv(OUT / 'okn_by_mo_egrokn.csv', encoding='utf-8-sig')
print(svod.to_string())

значение               федеральные  региональные  местные  итого
район2                                                          
Кош-Агачский район              30            20        0     50
Майминский район                 9            13        0     22
Онгудайский район               41            20        0     61
Турочакский район                3            12        0     15
Улаганский район                 9            12        0     21
Усть-Канский район               6            20        0     26
Усть-Коксинский район            1            26        0     27
Чемальский район                 9            15        0     24
Чойский район                    0             8        0      8
Шебалинский район                4            16        1     21
г. Горно-Алтайск                 3             7        0     10
район не определён               2             1        0      3
Итого                          117           170        1    288
